# Airbnb Market Intelligence — Guided EDA
**CodeAlpha Task 2 | Inside Airbnb, New York City**

Run from top to bottom after placing the detailed `listings.csv.gz` in `data/raw/`. Keep a record of the exact snapshot. Advertised prices are not realized revenue; availability is not occupancy; review counts are not bookings.

## 1. Load data and inspect its structure

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
TABLES = ROOT / "outputs" / "tables"; TABLES.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / "outputs" / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)
source = next((p for p in [RAW / "listings.csv.gz", RAW / "listings.csv"] if p.exists()), None)
if source is None:
    raise FileNotFoundError("Download detailed NYC listings.csv.gz into data/raw/.")
df = pd.read_csv(source, low_memory=False, compression="infer")
df.columns = df.columns.astype(str).str.strip()
print("Source:", source, "| Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Data quality: missingness, duplicates, and invalid prices

In [ ]:
missing = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct_non_null": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)
display(missing.head(25))
missing.to_csv(TABLES / "notebook_missingness_profile.csv")
print("Duplicate rows:", df.duplicated().sum())
if "id" in df:
    print("Duplicate listing IDs:", df["id"].duplicated().sum())
df["price_numeric"] = pd.to_numeric(
    df["price"].astype("string").str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)
print("Missing/unparseable price:", df["price_numeric"].isna().sum())
print("Non-positive prices:", (df["price_numeric"] <= 0).fillna(False).sum())

## 3. Price distribution and outlier sensitivity
Retain valid extreme values in summary statistics. Cap at the 99th percentile only for chart readability.

In [ ]:
analysis = df[df["price_numeric"].notna() & (df["price_numeric"] > 0)].copy()
p99 = analysis["price_numeric"].quantile(.99)
display(analysis["price_numeric"].describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_frame("price"))
print("P99:", p99, "| rows above P99:", (analysis["price_numeric"] > p99).sum())
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9,5))
sns.histplot(analysis.loc[analysis["price_numeric"] <= p99, "price_numeric"], bins=50, kde=True)
plt.title("Advertised Nightly Price Distribution (up to P99)")
plt.xlabel("Advertised nightly price (dataset currency)"); plt.ylabel("Listings")
plt.tight_layout(); plt.savefig(FIGURES / "notebook_price_distribution.png", dpi=150); plt.show()

## 4. Compare room types
Compare medians and sample sizes; group differences are descriptive, not causal.

In [ ]:
if "room_type" in analysis:
    room = analysis.groupby("room_type", dropna=False)["price_numeric"].agg(
        listing_count="count", median_price="median", mean_price="mean",
        q1=lambda x:x.quantile(.25), q3=lambda x:x.quantile(.75)
    ).sort_values("median_price", ascending=False)
    display(room); room.to_csv(TABLES / "notebook_price_by_room_type.csv")
    plot = analysis.copy(); plot["price_display"] = plot["price_numeric"].clip(upper=p99)
    plt.figure(figsize=(10,6)); sns.boxplot(data=plot, x="room_type", y="price_display")
    plt.title("Advertised Price by Room Type (capped at P99 for display)")
    plt.xlabel("Room type"); plt.ylabel("Nightly price (dataset currency)")
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(FIGURES / "notebook_price_by_room_type.png", dpi=150); plt.show()

## 5. Neighbourhood comparisons

In [ ]:
if "neighbourhood_group" in analysis:
    area = analysis.groupby("neighbourhood_group", dropna=False)["price_numeric"].agg(
        listing_count="count", median_price="median", mean_price="mean"
    ).sort_values("listing_count", ascending=False)
    display(area); area.to_csv(TABLES / "notebook_price_by_neighbourhood_group.csv")
    plt.figure(figsize=(10,6)); sns.barplot(data=area.reset_index(), x="neighbourhood_group", y="listing_count")
    plt.title("Listings by Neighbourhood Group")
    plt.xlabel("Neighbourhood group"); plt.ylabel("Listings with valid prices")
    plt.xticks(rotation=20); plt.tight_layout()
    plt.savefig(FIGURES / "notebook_listings_by_neighbourhood_group.png", dpi=150); plt.show()

## 6. Price and review activity
Spearman correlation is an exploratory association only. Reviews are not bookings; listing age and exposure may confound the relationship.

In [ ]:
if "number_of_reviews" in analysis:
    review_count = pd.to_numeric(analysis["number_of_reviews"], errors="coerce")
    pair = analysis[["price_numeric"]].join(review_count.rename("number_of_reviews")).dropna()
    rho = pair["price_numeric"].corr(pair["number_of_reviews"], method="spearman")
    print(f"Spearman rho: {rho:.3f}; sample size: {len(pair):,}")
    chart = pair[pair["price_numeric"] <= p99]
    plt.figure(figsize=(9,5)); sns.scatterplot(data=chart, x="price_numeric", y="number_of_reviews", alpha=.35, s=20)
    plt.title("Advertised Price vs Number of Reviews (price capped at P99)")
    plt.xlabel("Advertised nightly price (dataset currency)"); plt.ylabel("Number of reviews")
    plt.tight_layout(); plt.savefig(FIGURES / "notebook_price_vs_reviews.png", dpi=150); plt.show()

## 7. Availability and final interpretation

In [ ]:
if "availability_365" in analysis:
    availability = pd.to_numeric(analysis["availability_365"], errors="coerce").dropna()
    print(availability.describe())
    plt.figure(figsize=(9,5)); sns.histplot(availability, bins=30)
    plt.title("Reported Availability Across Listings")
    plt.xlabel("Availability days in next 365 days"); plt.ylabel("Listings")
    plt.tight_layout(); plt.savefig(FIGURES / "notebook_availability_distribution.png", dpi=150); plt.show()

## 8. Turn analysis into a professional report
1. Record the exact snapshot date and download date in `reports/data_source.md`.
2. Run `python src/run_eda.py` to validate the repeatable pipeline.
3. Use actual output tables to write 3–5 findings in `reports/findings_template.md`.
4. Include numerical evidence, business relevance, and a caveat for each finding.
5. Recommend next steps without claiming causality or inventing booking/revenue metrics.